# ST4 — Corporate CapEx: Data Ingestion & Cleaning
**Team 7 Lambda | Phase 1 | Feb 27 – Mar 2**

**Goal:** Pull annual CapEx, R&D, and Revenue figures for 10 companies from the SEC EDGAR XBRL API.
Compute intensity ratios and save a clean panel dataset.

**Companies (by sector):**
- Semiconductors: NVDA, INTC, AMD
- Hyperscalers: MSFT, GOOGL, AMZN
- Energy: XOM, NEE
- Mining/Critical Minerals: ALB, MP

**Causal role:** ST4 captures how firms restructure investment (CapEx / R&D) in response to
supply chain pressure (ST1) amplified by geopolitical risk (ST2) and energy policy asymmetries (ST3).

**Output:** `data/processed/st4_capex.parquet`

**API:** No key required. `User-Agent` header must be set (loaded from `.env` via `config.py`).

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
sys.path.append('..')

import time
import pandas as pd
import numpy as np
import requests
from pathlib import Path

from config import (
    DATA_PROC,
    CAPEX_COMPANIES,
    XBRL_CONCEPTS,
    START_YEAR,
    END_YEAR,
    SEC_USER_AGENT,
)
from utils import save_parquet, set_style, log

set_style()
log.info(f"ST4 setup complete. SEC User-Agent: '{SEC_USER_AGENT}'")

## 4A — SEC EDGAR XBRL API: Fetch Functions

**Endpoint pattern:**
```
GET https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json
```

The response contains an array of all historical filings for that concept.
We filter to `form=10-K` (annual) and `fp=FY` (full-year) to get one clean row per fiscal year.

**Rate limit:** EDGAR allows ~10 req/sec. We sleep 0.15 s between requests to stay polite.

In [ ]:
# ── EDGAR Fetch Helpers ─────────────────────────────────────────────────────

EDGAR_BASE    = "https://data.sec.gov/api/xbrl/companyconcept"
EDGAR_HEADERS = {"User-Agent": SEC_USER_AGENT}

# Revenue fallback chain — try in order if primary 'Revenues' concept returns nothing
REVENUE_FALLBACKS = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "SalesRevenueNet",
    "SalesRevenueGoodsNet",
]


def fetch_xbrl_concept(cik: str, concept: str, retries: int = 3) -> list:
    """
    Fetch all historical filings for one XBRL concept from SEC EDGAR.

    Args:
        cik:     10-digit CIK string with leading zeros (e.g. '0001045810')
        concept: us-gaap concept name (e.g. 'ResearchAndDevelopmentExpense')
        retries: number of retry attempts on transient failures

    Returns:
        List of filing dicts with keys: end, val, fy, fp, form, accn.
        Empty list on 404 (concept not filed by this company) or persistent error.

    Usage:
        rows = fetch_xbrl_concept('0001045810', 'ResearchAndDevelopmentExpense')
    """
    url = f"{EDGAR_BASE}/CIK{cik}/us-gaap/{concept}.json"

    for attempt in range(retries):
        try:
            r = requests.get(url, headers=EDGAR_HEADERS, timeout=30)
            if r.status_code == 404:
                log.warning(f"404 — concept not found: {concept} / CIK {cik}")
                return []
            r.raise_for_status()
            units = r.json().get("units", {})
            if "USD" not in units:
                log.warning(f"No USD units for {concept} / CIK {cik}")
                return []
            return units["USD"]
        except requests.exceptions.Timeout:
            log.warning(f"Timeout (attempt {attempt + 1}/{retries}): {concept} / CIK {cik}")
            time.sleep(2 ** attempt)
        except Exception as e:
            log.error(f"Error fetching {concept} / CIK {cik}: {e}")
            return []
    return []


def extract_annual(rows: list, start_year: int, end_year: int) -> pd.DataFrame:
    """
    Filter raw EDGAR filing rows to annual 10-K filings within the analysis window.

    Deduplicates by fiscal year — keeps the latest accession number to capture
    any amended filings (10-K/A).

    Args:
        rows:       list of dicts from fetch_xbrl_concept()
        start_year: inclusive lower bound (e.g. START_YEAR)
        end_year:   inclusive upper bound (e.g. END_YEAR)

    Returns:
        DataFrame with columns: fy (int), period_end (date str), val (float).
        Empty DataFrame if rows is empty or no annual filings found.

    Usage:
        annual_df = extract_annual(rows, START_YEAR, END_YEAR)
    """
    if not rows:
        return pd.DataFrame(columns=["fy", "period_end", "val"])

    df = pd.DataFrame(rows)

    # Keep full-year annual filings only
    if "form" in df.columns:
        df = df[df["form"].isin(["10-K", "10-K/A"])]
    if "fp" in df.columns:
        df = df[df["fp"] == "FY"]

    if df.empty:
        return pd.DataFrame(columns=["fy", "period_end", "val"])

    df["fy"]  = pd.to_numeric(df.get("fy"),  errors="coerce")
    df["val"] = pd.to_numeric(df.get("val"), errors="coerce")
    df = df[(df["fy"] >= start_year) & (df["fy"] <= end_year)]

    # Deduplicate: one row per fiscal year (latest accession wins)
    if "accn" in df.columns:
        df = df.sort_values("accn").drop_duplicates(subset=["fy"], keep="last")

    df = df.rename(columns={"end": "period_end"})
    keep = [c for c in ["fy", "period_end", "val"] if c in df.columns]
    return df[keep].reset_index(drop=True)

## 4B — Pull All Companies × Concepts

Iterates over every company in `CAPEX_COMPANIES` and fetches three concepts:
- `capex` → `PaymentsToAcquirePropertyPlantAndEquipment`
- `rd` → `ResearchAndDevelopmentExpense`
- `revenue` → `Revenues` (with fallback chain for companies that use alternate tags)

Results accumulate into a long-form DataFrame that is pivoted in the next cell.

In [ ]:
# ── Pull All Companies × Concepts from EDGAR ───────────────────────────────

records = []

for ticker, info in CAPEX_COMPANIES.items():
    cik    = info["cik"]
    sector = info["sector"]
    name   = info["name"]

    for label, concept in XBRL_CONCEPTS.items():
        if label == "ebitda":
            continue  # OperatingIncomeLoss is a proxy; skip for ingestion

        # Revenue gets a fallback chain; others use the concept directly
        concepts_to_try = REVENUE_FALLBACKS if label == "revenue" else [concept]

        fetched = pd.DataFrame()
        for c in concepts_to_try:
            log.info(f"Fetching {ticker} / {label} ({c})...")
            rows   = fetch_xbrl_concept(cik, c)
            annual = extract_annual(rows, START_YEAR, END_YEAR)
            if not annual.empty:
                fetched = annual
                break
            time.sleep(0.15)

        if fetched.empty:
            log.warning(f"No annual data found: {ticker} / {label}")
            continue

        fetched["ticker"]        = ticker
        fetched["sector"]        = sector
        fetched["concept_label"] = label
        records.append(fetched)
        time.sleep(0.15)  # polite EDGAR rate limiting

if records:
    raw_long = pd.concat(records, ignore_index=True)
    log.info(f"Long-form EDGAR data: {raw_long.shape} | "
             f"tickers: {raw_long['ticker'].nunique()} | "
             f"concepts: {raw_long['concept_label'].unique()}")
    display(raw_long.head(15))
else:
    log.warning(
        "No EDGAR data fetched. Possible causes:\n"
        "  1. SEC_USER_AGENT not set in .env (required by EDGAR)\n"
        "  2. Network/firewall issue\n"
        "  3. EDGAR API temporarily unavailable\n"
        "Creating empty stub — downstream cells will produce empty output."
    )
    raw_long = pd.DataFrame(columns=["fy", "period_end", "val", "ticker", "sector", "concept_label"])

In [ ]:
# ── Pivot Long → Wide and Compute Intensity Ratios ─────────────────────────

def build_st4_capex(long_df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot long-form EDGAR data into the st4_capex target schema and compute
    CapEx and R&D intensity ratios.

    Target schema:
        ticker, sector, period (fiscal year), capex (USD), rd_expense (USD),
        revenue (USD), capex_intensity (capex/revenue), rd_intensity (rd/revenue)

    Values are in USD as reported in SEC 10-K filings (not inflation-adjusted).
    Intensity ratios are undefined (NaN) where revenue is zero or missing.

    Args:
        long_df: output of the pull loop (columns: ticker, sector, fy, val, concept_label)

    Returns:
        Wide DataFrame in target schema. Empty stub if input is empty.

    Usage:
        st4_df = build_st4_capex(raw_long)
    """
    TARGET_COLS = [
        "ticker", "sector", "period",
        "capex", "rd_expense", "revenue",
        "capex_intensity", "rd_intensity",
    ]

    if long_df.empty or "concept_label" not in long_df.columns:
        return pd.DataFrame(columns=TARGET_COLS)

    # Pivot: index = ticker × sector × fy, columns = concept_label
    pivot = (
        long_df
        .pivot_table(
            index=["ticker", "sector", "fy"],
            columns="concept_label",
            values="val",
            aggfunc="first",   # deduplicated upstream; first == only
        )
        .reset_index()
        .rename(columns={"fy": "period", "rd": "rd_expense"})
    )
    pivot.columns.name = None

    # Ensure all value columns exist even if no data was fetched for them
    for col in ("capex", "rd_expense", "revenue"):
        if col not in pivot.columns:
            log.warning(f"Column '{col}' missing after pivot — filling with NaN.")
            pivot[col] = np.nan

    # Intensity ratios — guard against zero / missing revenue
    valid_rev = pivot["revenue"].gt(0)
    pivot["capex_intensity"] = np.where(valid_rev, pivot["capex"]    / pivot["revenue"], np.nan)
    pivot["rd_intensity"]    = np.where(valid_rev, pivot["rd_expense"] / pivot["revenue"], np.nan)

    pivot = pivot.sort_values(["ticker", "period"]).reset_index(drop=True)

    log.info(
        f"st4_capex built: {pivot.shape} | "
        f"tickers: {pivot['ticker'].nunique()} | "
        f"year range: {pivot['period'].min()}–{pivot['period'].max()}"
    )
    return pivot[TARGET_COLS]


st4_capex = build_st4_capex(raw_long)

if not st4_capex.empty:
    log.info("CapEx intensity by sector (mean across all years):")
    display(
        st4_capex.groupby("sector")[["capex_intensity", "rd_intensity"]]
        .mean()
        .round(4)
        .sort_values("capex_intensity", ascending=False)
    )
    log.info("Most recent year — all companies:")
    latest = st4_capex[st4_capex["period"] == st4_capex["period"].max()]
    display(latest[["ticker", "sector", "period", "capex_intensity", "rd_intensity"]].round(4))

In [ ]:
# ── Save ST4 Output to Parquet ──────────────────────────────────────────────

if not st4_capex.empty:
    save_parquet(st4_capex, DATA_PROC / "st4_capex.parquet", "ST4 CapEx & R&D")
else:
    log.warning("st4_capex is empty — saving stub so downstream notebooks don't crash.")
    stub = pd.DataFrame(columns=[
        "ticker", "sector", "period",
        "capex", "rd_expense", "revenue",
        "capex_intensity", "rd_intensity",
    ])
    save_parquet(stub, DATA_PROC / "st4_capex.parquet", "ST4 CapEx (stub)")

log.info("Phase 1 ST4 complete. Check data/processed/st4_capex.parquet.")